In [1]:
from typing import TypedDict, List
from typing_extensions import TypedDict

# define the AgentState class
# different attributes for different states of the agent
class AgentState(TypedDict):
    task: str             
    plan: str               
    draft: str              
    critique: str           
    content: List[str]      
    revision_number: int    
    max_revisions: int     

In [2]:
# node 1: PLan node
from langchain_core.messages import SystemMessage, HumanMessage

PLAN_PROMPT = """you are an AI assistant. You are tasked with writing an essay. You will write the essay in a step-by-step manner. You will start by planning the essay. You will write a plan for the essay. The plan will include the introduction, the body, and the conclusion. The plan will also include the thesis statement. The plan will also include the supporting evidence. The plan will also include the transitions between the paragraphs. The plan will also include the formatting of the essay. The plan will"""

def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state["task"])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

In [3]:
# node 2: research plan node
RESEARCH_PLAN_PROMPT = """you are an AI assistant who is helping a researcher to plan their research. The researcher has a task to write an essay on a specific topic. The task is to write an essay on the topic of """

from pydantic import BaseModel

# use structured output to get the queries
class Queries(BaseModel):
    queries: List[str]

def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state["task"])
    ])

    content = state.get("content", [])
    for q in queries.queries:
        results = tavily.search(q, max_results=2)
        for r in results["results"]:
            content.append(r["content"])

    return {"content": content}

In [4]:
# node 3: Generation node
WRITER_PROMPT = """you are a writer. You are given a plan and a draft. You are asked to generate a new draft based on the plan and the draft. You should not change the plan. You should not change the draft. You should not change the task. You should not change the revision number. You should not change the content. You should not change the user message. You should not change the system message. You should not change the model. You should not change the environment. You should not change
paln: {plan}
reference: {content}
"""

def generation_node(state: AgentState):
    content = "\n\n".join(state.get("content", []))

    user_message = HumanMessage(content=f"task:{state['task']}\n\n draft:\n{state.get('draft', '')}")

    messages = [
        SystemMessage(content=WRITER_PROMPT.format(plan=state["plan"], content=content)),
        user_message
    ]

    response = model.invoke(messages)

    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1
    }

In [5]:
#  node 4: reflection node
REFLECTION_PROMPT = """you are an AI assistant that is helping a writer to write an essay. You have just finished writing the essay and you are reflecting on it. Please provide a critique of the essay. The critique should be constructive and helpful. The critique should be in the form of a paragraph. The critique should be in the form of a paragraph. The critique should be in the form of a paragraph. The critique should be in the form of a paragraph. The critique should be in the form of a paragraph. The"""

def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state["draft"])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [6]:
# node 5: research Critique Node
RESEARCH_CRITIQUE_PROMPT = """
You are a researcher and depend on the critique suggestions, to search for more information about the topic. 
"""

def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state["critique"])
    ])

    content = state.get("content", [])
    for q in queries.queries:
        results = tavily.search(q, max_results=2)
        for r in results["results"]:
            content.append(r["content"])

    return {"content": content}

In [7]:
# the conditional statement
def should_continue(state: AgentState):
    if state["revision_number"] > state["max_revisions"]:
        return "end"
    return "reflect"


In [8]:
# combine the whole graph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

graph_builder = StateGraph(AgentState)

#  register all nodes
graph_builder.add_node("planner", plan_node)
graph_builder.add_node("research_plan", research_plan_node)
graph_builder.add_node("generate", generation_node)
graph_builder.add_node("reflect", reflection_node)
graph_builder.add_node("research_critique", research_critique_node)

# set up the entry point of the graph
graph_builder.set_entry_point("planner")

# fixed edge: planner -> research_plan
graph_builder.add_edge("planner", "research_plan")
graph_builder.add_edge("research_plan", "generate")
graph_builder.add_edge("reflect", "research_critique")
graph_builder.add_edge("research_critique", "generate")

# conditional edges: generate -> end or reflect
graph_builder.add_conditional_edges(
    "generate",
    should_continue,
    {"end": END, "reflect": "reflect"}
)

checkpointer = MemorySaver()
agent = graph_builder.compile(checkpointer=checkpointer)

In [9]:
# run the agent

config = {"configurable": {"thread_id": "essay-001"}}

for chunk in agent.stream({
    "task": "create an essay about how the AI change the software industry",
    "max_revisions": 2,
    "revision_number": 1,
}, config=config):
    print(chunk.keys())  # print the currently note

NameError: name 'model' is not defined